# Imports & env load

In [4]:
import chromadb
from langchain_chroma import Chroma
from groq import Groq
from dotenv import load_dotenv
import os 
from tqdm import tqdm
import time
from langchain_core.documents import Document

load_dotenv()
# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# client = Groq()

NVIDIA_API = os.getenv("NVIDIA_API_KEY")

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API
)

# Embedding Model

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Vaibhav Kesarwani\AppData\Local\Temp\ipykernel_9972\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
c:\Users\Vaibhav Kesarwani\Desktop\AI_Training_Batch_May_2026\submissions\assignments\assignment-03\vaibhav-kesarwani\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Model Selection

In [7]:
model_name = "meta/llama-3.1-8b-instruct"

# Hypothetical Questions

## Vector Database Connection

In [8]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [10]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [11]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'hypothetical_questions']

## Vector database retrieval

In [12]:
retriever = vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={
        'k': 3
    }
)

### Etracting the chunks from the database

In [13]:
collection = vectorstore_persisted._collection
total_docs = collection.count()

print(total_docs)

3337


## System message and user template to get the hypothetical questions

In [14]:
hypothetical_questions_system_message = """
You are improving retrieval for a RAG system.

Read ALL chunks together.

Generate ONLY 3 high-quality hypothetical questions
that could retrieve this ENTIRE batch.

NOT 3 per chunk.

Return ONLY:

Q1
Q2
Q3
"""

user_message_template = """
<Document>
{document}
</Document>
"""

## Processing the chunks to create the hypothetical questions in batch to overcome the rate limits

In [26]:
import json

batch_size = 20
hypothetical_questions = []

for batch_num, offset in enumerate(
    tqdm(range(0, total_docs, batch_size), desc="Processing batches"),
    start=1
):

    batch = collection.get(
        limit=batch_size,
        offset=offset,
        include=["documents"]
    )

    documents = batch["documents"]
    ids = batch["ids"]

    # print(f"\nProcessing batch {batch_num} ({len(documents)} docs)")

    batch_text = "\n\n".join(documents)

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "system",
                    "content": hypothetical_questions_system_message
                },
                {
                    "role": "user",
                    "content": user_message_template.format(document=batch_text)
                },
            ],
            temperature=0.2
        )

        questions = response.choices[0].message.content.strip()
        time.sleep(2)

    except Exception as e:
        print(f"Error in batch {batch_num}: {e}")
        continue

    
    question_list = [
        q.strip()
        for q in questions.split("\n")
        if q.strip()
    ][:3]   

    
    for q in question_list:

        hypothetical_questions.append(
            Document(
                page_content=q,
                metadata={
                    "batch": batch_num,
                    "offset": offset,
                    "parent_chunk_ids": json.dumps(ids),  
                    "parent_collection": "tesla-10k-2019-to-2023"
                }
            )
        )

Processing batches: 100%|██████████| 167/167 [10:47<00:00,  3.88s/it]


In [ ]:
len(hypothetical_questions)

In [28]:
hypothetical_questions

[Document(metadata={'batch': 1, 'offset': 0, 'parent_chunk_ids': '["text_0", "text_1", "text_2", "text_3", "text_4", "text_5", "text_6", "text_7", "text_8", "text_9", "text_10", "text_11", "text_12", "text_13", "text_14", "text_15", "text_16", "text_17", "text_18", "text_19"]', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content="Q1: What are the key attributes and qualifications of Tesla's Board of Directors, as outlined in the company's 10-K/A filing for the fiscal year ended December 31, 2021?"),
 Document(metadata={'batch': 1, 'offset': 0, 'parent_chunk_ids': '["text_0", "text_1", "text_2", "text_3", "text_4", "text_5", "text_6", "text_7", "text_8", "text_9", "text_10", "text_11", "text_12", "text_13", "text_14", "text_15", "text_16", "text_17", "text_18", "text_19"]', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content="Q2: How does the Board of Directors' composition and experience impact Tesla's corporate governance and decision-making processes, as described i

In [29]:
from collections import defaultdict
import json

grouped = defaultdict(list)

for doc in hypothetical_questions:
    batch = doc.metadata.get("batch")
    question = doc.page_content.strip()

    key = f"batch {batch}"
    grouped[key].append(question)

result = dict(grouped)

# save to json file
with open("hypothetical_questions.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

## Initalising the vector database for the hypothetical questions 

In [15]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [31]:
hypothetical_questions_vectorstore.delete_collection()

### Adding documents to the vector store

In [ ]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_questions
)

In [16]:
hypothetical_questions_vectorstore._collection.count()

501

### Retrieval from the vector database - hypothetical vector store

In [17]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

### User Query

In [24]:
user_query = "Who is the auditor for the registrant?"

In [18]:
benchmark_questions = {
    "HQ1": "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",
    "HQ2": "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",
    "HQ3": "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",
    "HQ4": "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?"
}

#### Inoking the user query

In [25]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

hypothetical_questions_retrieved

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='a3770079-f6ef-451b-aca6-8b7261b62df7', metadata={'batch': 36, 'offset': 700, 'parent_chunk_ids': '["text_700", "text_701", "text_702", "text_703", "text_704", "text_705", "text_706", "text_707", "text_708", "text_709", "text_710", "text_711", "text_712", "text_713", "text_714", "text_715", "text_716", "text_717", "text_718", "text_719"]', 'parent_collection': 'tesla-10k-2019-to-2023'}, page_content='Q1: What are the specific conditions under which the Borrower is required to provide audited financial statements to the Facility Agent, and what are the consequences of failing to do so?'),
 Document(id='ced114ba-30c1-43e2-8781-b2fa8763c192', metadata={'batch': 64, 'offset': 1260, 'parent_chunk_ids': '["text_1260", "text_1261", "text_1262", "text_1263", "text_1264", "text_1265", "text_1266", "text_1267", "text_1268", "text_1269", "text_1270", "text_1271", "text_1272", "text_1273", "text_1274", "text_1275", "text_1276", "text_1277", "text_1278", "text_1279"]', 'parent_collecti

### Fetching the retrived documents

In [26]:
import json

retrieved_docs = hypothetical_questions_retrieved

all_chunk_ids = set()

for d in retrieved_docs:
    all_chunk_ids.update(json.loads(d.metadata["parent_chunk_ids"]))

retrived_documents = vectorstore_persisted.get(
    ids=list(all_chunk_ids)
)

print(retrived_documents["documents"])

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


['6.\nLiquidation\tand\tBankruptcy\tEvents\nThe\tBorrower\thas\tnot\tentered\tinto\tany\tliquidation\tprocess,\tnor\tis\tthere\tany\tbankruptcy\tevent.\n24', 'Certain\tidentified\tinformation\thas\tbeen\tomitted\tfrom\tthis\tdocument\tbecause\tit\tis\tnot\tmaterial\tand\twould\tbe\tcompetitively\tharmful\tif\tpublicly\ndisclosed,\tand\thas\tbeen\tmarked\twith\t“[***]”\tto\tindicate\twhere\tomissions\thave\tbeen\tmade.\n\t\n\t\n7.\nInformation\nAll\twritten\tdocuments\tprovided\tby\tthe\tBorrower\tare\ttrue\tand\tvalid\tin\tall\tmaterial\taspects\tas\tof\tthe\tdate\tof\ndelivery\tof\tthe\tsame.\n\t\n8.\nMaterial\tLicenses\nThe\tBorrower\thas\tobtained\tmaterial\tlicenses\tin\trelation\tto\tthe\tProject\tpursuant\tto\tlaws\tand\tregulations\tof\tthe\nPRC.\n\t\n9.\nNo\tMaterial\tDefault\nTo\tthe\tBorrower’s\tknowledge,\tas\tof\tthe\tEffective\tDate\tof\tthis\tAgreement,\tthere\tis\tno\tmaterial\tdefault\tof\tthe\nBorrower\tunder\tany\tagreement\tto\twhich\tit\tis\ta\tparty\t(material\tis\

In [27]:
retrived_documents['documents']

['6.\nLiquidation\tand\tBankruptcy\tEvents\nThe\tBorrower\thas\tnot\tentered\tinto\tany\tliquidation\tprocess,\tnor\tis\tthere\tany\tbankruptcy\tevent.\n24',
 'Certain\tidentified\tinformation\thas\tbeen\tomitted\tfrom\tthis\tdocument\tbecause\tit\tis\tnot\tmaterial\tand\twould\tbe\tcompetitively\tharmful\tif\tpublicly\ndisclosed,\tand\thas\tbeen\tmarked\twith\t“[***]”\tto\tindicate\twhere\tomissions\thave\tbeen\tmade.\n\t\n\t\n7.\nInformation\nAll\twritten\tdocuments\tprovided\tby\tthe\tBorrower\tare\ttrue\tand\tvalid\tin\tall\tmaterial\taspects\tas\tof\tthe\tdate\tof\ndelivery\tof\tthe\tsame.\n\t\n8.\nMaterial\tLicenses\nThe\tBorrower\thas\tobtained\tmaterial\tlicenses\tin\trelation\tto\tthe\tProject\tpursuant\tto\tlaws\tand\tregulations\tof\tthe\nPRC.\n\t\n9.\nNo\tMaterial\tDefault\nTo\tthe\tBorrower’s\tknowledge,\tas\tof\tthe\tEffective\tDate\tof\tthis\tAgreement,\tthere\tis\tno\tmaterial\tdefault\tof\tthe\nBorrower\tunder\tany\tagreement\tto\twhich\tit\tis\ta\tparty\t(material\tis

## Final system message and the user template

In [28]:
final_query_system = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [29]:
final_query_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

### Combining the system and user prompt

In [30]:
prompt = [
    {
        "role": "system",
        "content": final_query_system
    },
    {
        "role": "user",
        "content": final_query_user_message_template.format(
            context="\n".join(retrived_documents['documents']),
            question=user_query
        )
    }
]

## Output 

In [30]:
model_name="meta/llama-3.1-8b-instruct"

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = e

print(prediction)

PricewaterhouseCoopers LLP.


In [31]:
import json

def build_final_schema(
    question_id,
    user_query,
    hypothetical_questions_retrieved,
    retrieved_docs,
    final_answer,
    comparison_with_baseline=""
):
    
    # -------------------------
    # 1. Hypothetical questions formatting
    # -------------------------
    hypo_list = []

    for d in hypothetical_questions_retrieved:
        hypo_list.append({
            "hypothetical_question": d.page_content,
            "parent_chunk_id": json.loads(d.metadata["parent_chunk_ids"]),
            "section": d.metadata.get("section", "unknown"),
            "year": d.metadata.get("year", 2025),
            "score": float(d.metadata.get("score", 0.0))
        })

    # -------------------------
    # 2. Parent chunks formatting
    # -------------------------
    parent_chunks = []

    for chunk_id, doc_text in zip(
        retrieved_docs.get("ids", []),
        retrieved_docs.get("documents", [])
    ):
        parent_chunks.append({
            "chunk_id": chunk_id,
            "source_doc": "tesla-10k-2019-to-2023",
            "section": "10-K context",
            "year": 2025
        })

    # -------------------------
    # 3. Final schema
    # -------------------------
    output = {
        "question_id": question_id,
        "user_query": user_query,
        "retrieved_hypothetical_questions": hypo_list,
        "parent_chunks_used": parent_chunks,
        "final_answer": final_answer,
        "citations": [],
        "comparison_with_baseline": comparison_with_baseline
    }

    return output

In [ ]:
final_output = build_final_schema(
    question_id="HQ1",
    user_query=user_query,
    hypothetical_questions_retrieved=hypothetical_questions_retrieved,
    retrieved_docs=retrived_documents,
    final_answer=prediction,
    comparison_with_baseline="PricewaterhouseCoopers LLP."
)

print(json.dumps(final_output, indent=2))

{
  "question_id": "HQ1",
  "user_query": "Who is the auditor for the registrant?",
  "retrieved_hypothetical_questions": [
    {
      "hypothetical_question": "Q1: What are the specific conditions under which the Borrower is required to provide audited financial statements to the Facility Agent, and what are the consequences of failing to do so?",
      "parent_chunk_id": [
        "text_700",
        "text_701",
        "text_702",
        "text_703",
        "text_704",
        "text_705",
        "text_706",
        "text_707",
        "text_708",
        "text_709",
        "text_710",
        "text_711",
        "text_712",
        "text_713",
        "text_714",
        "text_715",
        "text_716",
        "text_717",
        "text_718",
        "text_719"
      ],
      "section": "unknown",
      "year": 2025,
      "score": 0.0
    },
    {
      "hypothetical_question": "Q3: What are the key filing dates and descriptions of the various exhibits and supplemental indenture

In [32]:
all_results = []

for qid, query in benchmark_questions.items():

    print(f"\nRunning {qid}...")

    # 1. Retrieve hypothetical questions
    hypo_docs = hypothetical_questions_vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={'k': 5}
    ).invoke(query)

    # 2. Extract parent chunks
    all_chunk_ids = set()
    for d in hypo_docs:
        all_chunk_ids.update(json.loads(d.metadata["parent_chunk_ids"]))

    retrieved_docs = vectorstore_persisted.get(
        ids=list(all_chunk_ids)
    )

    # 3. Generate final answer
    prompt = [
        {"role": "system", "content": final_query_system},
        {
            "role": "user",
            "content": final_query_user_message_template.format(
                context="\n".join(retrieved_docs["documents"]),
                question=query
            )
        }
    ]

    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    answer = response.choices[0].message.content.strip()

    # 4. Build final schema
    final_output = build_final_schema(
        question_id=qid,
        user_query=query,
        hypothetical_questions_retrieved=hypo_docs,
        retrieved_docs=retrieved_docs,
        final_answer=answer,
        comparison_with_baseline="Baseline vs HyDE retrieval comparison pending manual evaluation."
    )

    all_results.append(final_output)


Running HQ1...

Running HQ2...

Running HQ3...

Running HQ4...


In [34]:
with open("hypothetical_final_output.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

In [33]:
with open("benchmark_hypothetical_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)